
# Project 1 ART Data Exploration

HSLU NLP HS26 Abductive Commonsense Reasoning

## Objectives
- Explore the ART dataset
- Understand the input format and labels
- Analyze text lengths and vocabulary
- Prepare preprocessing decisions for MLP and GRU models


In [1]:
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence

# Load the HSLU ART dataset
dataset = load_dataset("amarfurt/art-nlp-hs26")

# Show dataset structure
print(dataset)

# Inspect column names
print("\nColumns:", dataset["train"].column_names)

# Inspect one training example
print("\nFirst example:")
print(dataset["train"][0])


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 16801
    })
    validation: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
        num_rows: 1532
    })
})

Columns: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label']

First example:
{'observation_1': 'A baby panda was lost in the woods.', 'observation_2': 'He was reunited with his parents, who also lived at the zoo.', 'hypothesis_1': 'He was never lost.', 'hypothesis_2': 'All the people from the zoo looked for him.', 'label': 2}


### Data Format Cleaning

The ART training dataset was scanned for HTML tags, URLs, HTML entities, emojis, invisible characters, and Unicode irregularities.

No HTML, URLs, emojis, or invisible characters were found. Only a few encoding issues were identified.

We normalize apostrophe variants and repair clearly identifiable corrupted apostrophes during tokenization. Other corrupted characters are left unchanged because their original values cannot be recovered reliably.

No examples are removed, and the original dataset remains unchanged. All preprocessing decisions are based on the training split.

In [2]:
import re
import unicodedata
from collections import Counter

# Check training data for formatting issues
fields = [
    "observation_1",
    "observation_2",
    "hypothesis_1",
    "hypothesis_2"
]

patterns = {
    "HTML tags": r"</?[a-zA-Z][^>]*>",
    "URLs": r"https?://\S+|www\.\S+",
    "HTML entities": r"&(?:\w+|#\d+|#x[0-9a-fA-F]+);",
    "Emojis": r"[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "Invisible characters": r"[\u200B-\u200F\uFEFF]",
    "Replacement characters": r"\uFFFD"
}

counts = Counter()
examples = {}
unicode_chars = Counter()

for example in dataset["train"]:
    for field in fields:
        text = example[field]

        for name, pattern in patterns.items():
            if re.search(pattern, text):
                counts[name] += 1
                examples.setdefault(name, text)

        # Identify non-ASCII characters
        unicode_chars.update(
            char for char in text if ord(char) > 127
        )

print("=== FORMAT CLEANING SCAN ===")

for name in patterns:
    print(f"{name}: {counts[name]}")

    if name in examples:
        print(f"  Example: {examples[name][:120]}")

print("\n=== NON-ASCII CHARACTERS ===")

for char, count in unicode_chars.most_common(20):
    name = unicodedata.name(char, "UNKNOWN")
    print(f"{repr(char)} ({name}): {count}")


=== FORMAT CLEANING SCAN ===
HTML tags: 0
URLs: 0
HTML entities: 0
Emojis: 0
Invisible characters: 0
Replacement characters: 3
  Example: Jo��o was frustrated at his country's government.

=== NON-ASCII CHARACTERS ===
'’' (RIGHT SINGLE QUOTATION MARK): 32
'�' (REPLACEMENT CHARACTER): 8
'é' (LATIN SMALL LETTER E WITH ACUTE): 2
'´' (ACUTE ACCENT): 1


In [3]:
# Inspect texts containing unusual characters
problem_chars = ["\uFFFD", "\u00B4"]

for idx, example in enumerate(dataset["train"]):
    for field in fields:
        text = example[field]

        if any(char in text for char in problem_chars):
            print(f"Row: {idx}")
            print(f"Field: {field}")
            print(f"Text: {text}")
            print("-" * 50)


Row: 8253
Field: observation_1
Text: Jo��o was frustrated at his country's government.
--------------------------------------------------
Row: 10069
Field: observation_2
Text: He opened the envelope and was disappointed he hadn���t been accepted.
--------------------------------------------------
Row: 14558
Field: hypothesis_2
Text: I was hired, but I was working around very old people, and I didn´t like it.
--------------------------------------------------
Row: 15363
Field: observation_2
Text: Alex took his clothes to his Mom���s house for her to clean.
--------------------------------------------------


In [4]:
import pandas as pd

# Check label distribution for each split
for split in ["train", "validation"]:
    labels = pd.Series(dataset[split]["label"])

    print(f"\n{split.upper()}")
    print(labels.value_counts().sort_index())
    print("\nPercentages:")
    print(labels.value_counts(normalize=True).sort_index() * 100)



TRAIN
1    8461
2    8340
Name: count, dtype: int64

Percentages:
1    50.360098
2    49.639902
Name: proportion, dtype: float64

VALIDATION
1    508
2    492
Name: count, dtype: int64

Percentages:
1    50.8
2    49.2
Name: proportion, dtype: float64


In [5]:
import re
import pandas as pd

fields = [
    "observation_1",
    "observation_2",
    "hypothesis_1",
    "hypothesis_2"
]

# Simple word-level tokenizer (preliminary)
def tokenize(text):
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text.lower())

# Calculate token lengths using training data only
lengths = []

for example in dataset["train"]:
    segment_lengths = [
        len(tokenize(example[field]))
        for field in fields
    ]

    # Add 3 separator tokens
    total_length = sum(segment_lengths) + 3
    lengths.append(segment_lengths + [total_length])

length_df = pd.DataFrame(
    lengths,
    columns=["O1", "O2", "H1", "H2", "Total"]
)

# Show statistics
print(length_df.describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).round(2))


             O1        O2        H1        H2     Total
count  16801.00  16801.00  16801.00  16801.00  16801.00
mean       9.14     10.48      9.48      9.50     41.61
std        2.60      2.86      3.13      3.16      7.52
min        3.00      3.00      1.00      1.00     20.00
50%        9.00     11.00      9.00      9.00     41.00
90%       13.00     14.00     13.00     13.00     51.00
95%       14.00     15.00     15.00     15.00     55.00
99%       15.00     16.00     20.00     20.00     64.00
max       19.00     20.00     35.00     34.00     92.00


In [6]:
from collections import Counter

# Build vocabulary using training data only
word_counts = Counter()

for example in dataset["train"]:
    for field in fields:
        tokens = tokenize(example[field])
        word_counts.update(tokens)

# Vocabulary statistics
total_tokens = sum(word_counts.values())
vocab_size = len(word_counts)

rare_words = sum(
    1 for count in word_counts.values()
    if count == 1
)

print("Total tokens:", total_tokens)
print("Vocabulary size:", vocab_size)
print("Words occurring only once:", rare_words)
print("Percentage of vocabulary occurring once:",
      round(rare_words / vocab_size * 100, 2), "%")

print("\n20 most common tokens:")
print(word_counts.most_common(20))


Total tokens: 648679
Vocabulary size: 16752
Words occurring only once: 5197
Percentage of vocabulary occurring once: 31.02 %

20 most common tokens:
[('.', 65608), ('the', 27649), ('to', 23083), ('a', 20893), ('was', 15793), ('and', 13201), ('her', 9464), ('he', 8280), ('his', 8122), ('i', 7536), ('she', 6778), ('in', 6710), (',', 6077), ('for', 6046), ('it', 5826), ('of', 5700), ('had', 5531), ('on', 4765), ('at', 3795), ('my', 3694)]


In [7]:
SEP = "[SEP]"

def prepare_example(example):
    tokens = []

    for i, field in enumerate(fields):
        if i > 0:
            tokens.append(SEP)

        tokens.extend(tokenize(example[field]))

    label = example["label"] - 1

    return {
        "tokens": tokens,
        "label": label
    }


# Test with the first training example
example = prepare_example(dataset["train"][0])

print("Input:")
print(" ".join(example["tokens"]))

print("\nSequence length:", len(example["tokens"]))
print("Label:", example["label"])
print("Correct hypothesis:", example["label"] + 1)


Input:
a baby panda was lost in the woods . [SEP] he was reunited with his parents , who also lived at the zoo . [SEP] he was never lost . [SEP] all the people from the zoo looked for him .

Sequence length: 41
Label: 1
Correct hypothesis: 2


In [8]:
import gensim.downloader as api

model_name = "fasttext-wiki-news-subwords-300"

# Read metadata (does not download the model)
info = api.info(model_name)

print("Model:", model_name)
print("Dimensions:", info["parameters"]["dimension"])
print("Vocabulary size:", info["num_records"])
print("Download size:", round(info["file_size"] / 1024**3, 2), "GiB")
print("Training corpus:", info["base_dataset"])


Model: fasttext-wiki-news-subwords-300
Dimensions: 300
Vocabulary size: 999999
Download size: 0.94 GiB
Training corpus: Wikipedia 2017, UMBC webbase corpus and statmt.org news dataset (16B tokens)


In [9]:
import gensim.downloader as api

# Download and load pretrained fastText embeddings
fasttext = api.load("fasttext-wiki-news-subwords-300")

print("Model loaded successfully!")
print("Vocabulary size:", len(fasttext))
print("Embedding dimensions:", fasttext.vector_size)

# Inspect a word embedding
word = "panda"

print(f"\nEmbedding for '{word}' (first 10 values):")
print(fasttext[word][:10])


Model loaded successfully!
Vocabulary size: 999999
Embedding dimensions: 300

Embedding for 'panda' (first 10 values):
[ 0.082003 -0.015127 -0.053782 -0.079441 -0.087394 -0.033851  0.045585
 -0.15176  -0.06694   0.064976]


In [10]:
# Check which ART tokens have pretrained fastText vectors
known_words = [
    word for word in word_counts
    if word in fasttext.key_to_index
]

unknown_words = [
    word for word in word_counts
    if word not in fasttext.key_to_index
]

# Vocabulary coverage
vocab_coverage = len(known_words) / len(word_counts) * 100

# Token coverage (weighted by frequency)
known_tokens = sum(word_counts[word] for word in known_words)
token_coverage = known_tokens / total_tokens * 100

print(f"Vocabulary size: {len(word_counts):,}")
print(f"Known words: {len(known_words):,}")
print(f"Unknown words: {len(unknown_words):,}")
print(f"Vocabulary coverage: {vocab_coverage:.2f}%")

print(f"\nToken coverage: {token_coverage:.2f}%")

# Most frequent unknown tokens
print("\n20 most frequent unknown tokens:")
print(
    sorted(
        [(word, word_counts[word]) for word in unknown_words],
        key=lambda x: x[1],
        reverse=True
    )[:20]
)


Vocabulary size: 16,752
Known words: 15,007
Unknown words: 1,745
Vocabulary coverage: 89.58%

Token coverage: 98.40%

20 most frequent unknown tokens:
[("didn't", 1604), ("couldn't", 602), ("wasn't", 437), ("gina's", 194), ("wouldn't", 130), ("year's", 100), ("doesn't", 92), ("hadn't", 88), ("amy's", 85), ("friend's", 85), ("bob's", 68), ("i'm", 65), ("he'd", 61), ("don't", 60), ("it's", 58), ("can't", 56), ("tim's", 56), ("tom's", 54), ("she'd", 52), ("mary's", 50)]


In [11]:
import re
from collections import Counter

def tokenize_v2(text):
    text = text.lower()

    # Normalize apostrophe variants
    text = text.replace("’", "'").replace("´", "'")

    # Repair clearly corrupted apostrophes
    text = re.sub(r"(?<=\w)\uFFFD+(?=[st]\b)", "'", text)

    # Irregular negative contractions
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bshan't\b", "shall not", text)

    # Regular negative contractions
    text = re.sub(r"\b([a-z]+)n't\b", r"\1 not", text)

    # Unambiguous positive contractions
    text = re.sub(r"\b([a-z]+)'m\b", r"\1 am", text)
    text = re.sub(r"\b([a-z]+)'re\b", r"\1 are", text)
    text = re.sub(r"\b([a-z]+)'ve\b", r"\1 have", text)
    text = re.sub(r"\b([a-z]+)'ll\b", r"\1 will", text)

    tokens = re.findall(r"\w+(?:'\w+)?|[^\w\s]", text)

    # Forms that may represent verb contractions
    ambiguous = {
        "it's", "he's", "she's", "that's",
        "there's", "here's", "what's",
        "who's", "where's", "how's",
        "let's"
    }

    result = []

    for token in tokens:
        if token.endswith("'s") and token not in ambiguous:
            # Preserve the base word and mark possession
            result.extend([token[:-2], "possessive"])
        else:
            result.append(token)

    return result


test_sentences = [
    "He hadn���t been accepted.",
    "I didn´t like it.",
    "Alex went to his Mom���s house.",
    "Jo��o was frustrated."
]

for sentence in test_sentences:
    print(sentence)
    print("->", tokenize_v2(sentence))
    print()



# Test our improved tokenizer
examples = [
    "John didn't go to school.",
    "She couldn't find her keys.",
    "He wasn't hungry.",
    "I can't believe it."
]

for sentence in examples:
    print(sentence)
    print("->", tokenize_v2(sentence))
    print()


He hadn���t been accepted.
-> ['he', 'had', 'not', 'been', 'accepted', '.']

I didn´t like it.
-> ['i', 'did', 'not', 'like', 'it', '.']

Alex went to his Mom���s house.
-> ['alex', 'went', 'to', 'his', 'mom', 'possessive', 'house', '.']

Jo��o was frustrated.
-> ['jo', '�', '�', 'o', 'was', 'frustrated', '.']

John didn't go to school.
-> ['john', 'did', 'not', 'go', 'to', 'school', '.']

She couldn't find her keys.
-> ['she', 'could', 'not', 'find', 'her', 'keys', '.']

He wasn't hungry.
-> ['he', 'was', 'not', 'hungry', '.']

I can't believe it.
-> ['i', 'can', 'not', 'believe', 'it', '.']



In [12]:
# Check that the possessive marker has an embedding
assert "possessive" in fasttext.key_to_index

test_sentences = [
    "Gina's friend was happy.",
    "John's car was stolen.",
    "His mother's house was empty.",
    "It's raining outside.",
    "He'd already left.",
    "She couldn't find her keys."
]

for sentence in test_sentences:
    tokens = tokenize_v2(sentence)

    unknown = [
        token for token in tokens
        if token not in fasttext.key_to_index
    ]

    print(sentence)
    print("Tokens:", tokens)
    print("Unknown:", unknown)
    print()


Gina's friend was happy.
Tokens: ['gina', 'possessive', 'friend', 'was', 'happy', '.']
Unknown: []

John's car was stolen.
Tokens: ['john', 'possessive', 'car', 'was', 'stolen', '.']
Unknown: []

His mother's house was empty.
Tokens: ['his', 'mother', 'possessive', 'house', 'was', 'empty', '.']
Unknown: []

It's raining outside.
Tokens: ["it's", 'raining', 'outside', '.']
Unknown: ["it's"]

He'd already left.
Tokens: ["he'd", 'already', 'left', '.']
Unknown: ["he'd"]

She couldn't find her keys.
Tokens: ['she', 'could', 'not', 'find', 'her', 'keys', '.']
Unknown: []



In [13]:
test_sentences = [
    "I'm going home.",
    "I've lost my keys.",
    "We're going shopping.",
    "They'll arrive tomorrow.",
    "It's raining.",
    "He'd already left.",
    "John's car was stolen.",
    "She couldn't find her keys."
]

for sentence in test_sentences:
    print(sentence)
    print("->", tokenize_v2(sentence))
    print()


I'm going home.
-> ['i', 'am', 'going', 'home', '.']

I've lost my keys.
-> ['i', 'have', 'lost', 'my', 'keys', '.']

We're going shopping.
-> ['we', 'are', 'going', 'shopping', '.']

They'll arrive tomorrow.
-> ['they', 'will', 'arrive', 'tomorrow', '.']

It's raining.
-> ["it's", 'raining', '.']

He'd already left.
-> ["he'd", 'already', 'left', '.']

John's car was stolen.
-> ['john', 'possessive', 'car', 'was', 'stolen', '.']

She couldn't find her keys.
-> ['she', 'could', 'not', 'find', 'her', 'keys', '.']



In [14]:

from collections import Counter

remaining = Counter()

for example in dataset["train"]:
    for field in fields:
        tokens = tokenize_v2(example[field])

        remaining.update(
            token for token in tokens if "'" in token
        )

print("Remaining contractions / possessives:")
print("Unique tokens:", len(remaining))
print("Total occurrences:", sum(remaining.values()))

print("\n30 most frequent:")
for word, count in remaining.most_common(30):
    in_fasttext = word in fasttext.key_to_index
    print(f"{word:20} {count:5}  fastText: {in_fasttext}")


Remaining contractions / possessives:
Unique tokens: 26
Total occurrences: 400

30 most frequent:
'                       82  fastText: True
he'd                    61  fastText: False
it's                    58  fastText: False
she'd                   52  fastText: False
he's                    37  fastText: False
i'd                     25  fastText: False
she's                   19  fastText: False
they'd                  14  fastText: False
that's                  13  fastText: False
there's                  6  fastText: False
s'mores                  4  fastText: False
who'd                    3  fastText: False
o'clock                  3  fastText: False
we'd                     3  fastText: False
who's                    3  fastText: False
what's                   3  fastText: False
did't                    2  fastText: False
d'oeuvres                2  fastText: False
could't                  2  fastText: False
it'd                     2  fastText: False
boo'd                  

In [15]:
word_counts_v2 = Counter()

for example in dataset["train"]:
    for field in fields:
        word_counts_v2.update(
            tokenize_v2(example[field])
        )

known_v2 = [
    word for word in word_counts_v2
    if word in fasttext.key_to_index
]

total_v2 = sum(word_counts_v2.values())

vocab_coverage_v2 = (
    len(known_v2) / len(word_counts_v2) * 100
)

token_coverage_v2 = (
    sum(word_counts_v2[w] for w in known_v2)
    / total_v2 * 100
)

print(f"Old vocabulary coverage: {vocab_coverage:.2f}%")
print(f"New vocabulary coverage: {vocab_coverage_v2:.2f}%")

print(f"\nOld token coverage: {token_coverage:.2f}%")
print(f"New token coverage: {token_coverage_v2:.2f}%")

# Examine remaining unknown words
unknown_v2 = [
    word for word in word_counts_v2
    if word not in fasttext.key_to_index
]

print("\nMost frequent remaining unknown tokens:")
print(sorted(
    [(w, word_counts_v2[w]) for w in unknown_v2],
    key=lambda x: x[1],
    reverse=True
)[:20])


Old vocabulary coverage: 89.58%
New vocabulary coverage: 95.03%

Old token coverage: 98.40%
New token coverage: 99.62%

Most frequent remaining unknown tokens:
[("he'd", 61), ("it's", 58), ("she'd", 52), ("he's", 37), ('kelsi', 31), ('marcy', 31), ('rhonda', 30), ("i'd", 25), ('joanie', 23), ('mindy', 22), ('mturk', 21), ('horace', 21), ('tabitha', 21), ('hallie', 21), ('horatio', 21), ('juanita', 20), ('callie', 19), ("she's", 19), ('briana', 18), ('cornelia', 17)]


In [16]:
# Special tokens
PAD = "[PAD]"
UNK = "[UNK]"
SEP = "[SEP]"

# Initialize vocabulary
vocab = {
    PAD: 0,
    UNK: 1,
    SEP: 2
}

# Add training vocabulary covered by fastText
for word, count in word_counts_v2.most_common():
    if word in fasttext.key_to_index:
        vocab[word] = len(vocab)

print("Final vocabulary size:", len(vocab))

print("\nSpecial tokens:")
for token in [PAD, UNK, SEP]:
    print(f"{token}: {vocab[token]}")

print("\nExample word IDs:")
for word in ["panda", "lost", "woods", "reunited"]:
    print(f"{word}: {vocab.get(word, vocab[UNK])}")


Final vocabulary size: 15038

Special tokens:
[PAD]: 0
[UNK]: 1
[SEP]: 2

Example word IDs:
panda: 3385
lost: 149
woods: 747
reunited: 5026


In [17]:
# Analyze remaining unknown contractions
unknown_contractions = {
    word: count
    for word, count in word_counts_v2.items()
    if "'" in word and word not in fasttext.key_to_index
}

print("=== REMAINING UNKNOWN CONTRACTIONS ===")
print("Unique:", len(unknown_contractions))
print("Total occurrences:", sum(unknown_contractions.values()))

print("\n20 most frequent:")
for word, count in sorted(
    unknown_contractions.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]:
    print(f"{word:20} {count:5}")


=== REMAINING UNKNOWN CONTRACTIONS ===
Unique: 25
Total occurrences: 318

20 most frequent:
he'd                    61
it's                    58
she'd                   52
he's                    37
i'd                     25
she's                   19
they'd                  14
that's                  13
there's                  6
s'mores                  4
who'd                    3
o'clock                  3
we'd                     3
who's                    3
what's                   3
did't                    2
d'oeuvres                2
could't                  2
it'd                     2
boo'd                    1


In [18]:
def encode_example(example):
    tokens = []

    for i, field in enumerate(fields):
        if i > 0:
            tokens.append(SEP)

        tokens.extend(tokenize_v2(example[field]))

    # Convert tokens to vocabulary IDs
    input_ids = [
        vocab.get(token, vocab[UNK])
        for token in tokens
    ]

    # Convert label from {1, 2} to {0, 1}
    label = example["label"] - 1

    return {
        "input_ids": input_ids,
        "label": label
    }


encoded = encode_example(dataset["train"][0])

print("Token IDs:")
print(encoded["input_ids"])

print("\nSequence length:", len(encoded["input_ids"]))
print("Label:", encoded["label"])
print("Number of unknown tokens:",
      encoded["input_ids"].count(vocab[UNK]))


Token IDs:
[6, 231, 3385, 7, 149, 14, 4, 747, 3, 2, 10, 7, 5026, 25, 11, 140, 15, 191, 521, 463, 23, 4, 603, 3, 2, 10, 7, 67, 149, 3, 2, 40, 4, 188, 47, 4, 603, 168, 16, 38, 3]

Sequence length: 41
Label: 1
Number of unknown tokens: 0


In [19]:
import numpy as np
import torch
import torch.nn as nn

embedding_dim = fasttext.vector_size

# Initialize embedding matrix
embedding_matrix = np.zeros(
    (len(vocab), embedding_dim),
    dtype=np.float32
)

# Copy pretrained fastText vectors
for word, idx in vocab.items():
    if word in [PAD, UNK, SEP]:
        continue

    embedding_matrix[idx] = fasttext[word]

# [UNK]: mean of the available word embeddings
embedding_matrix[vocab[UNK]] = embedding_matrix[3:].mean(axis=0)

# [SEP]: fixed random vector for separating text segments
rng = np.random.default_rng(42)
embedding_matrix[vocab[SEP]] = rng.normal(
    0, 0.05, embedding_dim
)

# [PAD] remains a zero vector

# Create frozen PyTorch embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    torch.from_numpy(embedding_matrix),
    freeze=True,
    padding_idx=vocab[PAD]
)

print("Embedding matrix shape:", embedding_layer.weight.shape)
print("Trainable:", embedding_layer.weight.requires_grad)
print("PAD vector is zero:",
      torch.all(embedding_layer.weight[vocab[PAD]] == 0).item())


Embedding matrix shape: torch.Size([15038, 300])
Trainable: False
PAD vector is zero: True


In [20]:
# Convert our example IDs into a PyTorch tensor
input_ids = torch.tensor(
    encoded["input_ids"],
    dtype=torch.long
)

# Look up the pretrained embeddings
vectors = embedding_layer(input_ids)

print("Input IDs shape:", input_ids.shape)
print("Embedding output shape:", vectors.shape)


Input IDs shape: torch.Size([41])
Embedding output shape: torch.Size([41, 300])


In [21]:
from torch.utils.data import Dataset, DataLoader

class ARTDataset(Dataset):
    def __init__(self, hf_dataset):
        # Encode all examples once
        self.examples = [
            encode_example(example)
            for example in hf_dataset
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# Create PyTorch datasets
train_dataset = ARTDataset(dataset["train"])
val_dataset = ARTDataset(dataset["validation"])

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

print("\nFirst encoded example:")
print(train_dataset[0])


Training examples: 16801
Validation examples: 1000

First encoded example:
{'input_ids': [6, 231, 3385, 7, 149, 14, 4, 747, 3, 2, 10, 7, 5026, 25, 11, 140, 15, 191, 521, 463, 23, 4, 603, 3, 2, 10, 7, 67, 149, 3, 2, 40, 4, 188, 47, 4, 603, 168, 16, 38, 3], 'label': 1}


In [22]:
def collate_fn(batch):
    """
    Pad sequences dynamically to the longest
    sequence in the batch.
    """

    sequences = [
        torch.tensor(item["input_ids"], dtype=torch.long)
        for item in batch
    ]

    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0  # [PAD] token
    )

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    return {
        "input_ids": padded_sequences,
        "lengths": lengths,
        "labels": labels
    }


In [23]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

# Inspect one training batch
batch = next(iter(train_loader))

print("Input IDs shape:", batch["input_ids"].shape)
print("Lengths shape:", batch["lengths"].shape)
print("Labels shape:", batch["labels"].shape)

print("\nFirst 5 sequence lengths:")
print(batch["lengths"][:5])

print("\nLabels:")
print(batch["labels"][:10])

print("\nPadding tokens in batch:",
      (batch["input_ids"] == vocab[PAD]).sum().item())


Input IDs shape: torch.Size([32, 57])
Lengths shape: torch.Size([32])
Labels shape: torch.Size([32])

First 5 sequence lengths:
tensor([45, 46, 43, 37, 40])

Labels:
tensor([1, 1, 0, 0, 0, 1, 0, 1, 1, 1])

Padding tokens in batch: 538


In [24]:
import sys
from pathlib import Path

# Make the repository root importable
repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from project1.src.preprocessing import build_vocab, encode_example

# Build vocabulary using our reusable function
new_vocab = build_vocab(dataset["train"], fasttext)

print("Vocabulary size:", len(new_vocab))
print("Matches original vocabulary:", new_vocab == vocab)

# Encode first training example
new_encoded = encode_example(dataset["train"][0], new_vocab)

print("Matches original encoding:", new_encoded == encoded)
print("Sequence length:", len(new_encoded["input_ids"]))
print("Label:", new_encoded["label"])


Vocabulary size: 15038
Matches original vocabulary: True
Matches original encoding: True
Sequence length: 41
Label: 1


## Final Preprocessing Decisions

The following preprocessing decisions are used for both the
MLP baseline and the upcoming GRU model.

| Step | Decision | Reason |
|---|---|---|
| Tokenization | Regex-based word-level tokenizer | Simple and compatible with pretrained fastText embeddings |
| Lowercasing | Yes | Reduces vocabulary size and improves embedding coverage |
| Contractions | Expand unambiguous forms and negations | Preserves important meaning while reducing unknown tokens |
| Possessives | Split most `'s` forms into base word + "possessive" | Reduces OOV tokens while retaining a possession marker |
| Stopwords | Keep | Words such as "not", "before", and "after" are important for reasoning |
| Punctuation | Keep as separate tokens | May provide useful sentence structure information |
| Stemming | No | Can remove useful linguistic information |
| Lemmatization | No | Not necessary for this baseline; preserves original word forms |
| Rare words | Keep if covered by fastText | Pretrained embeddings are available even for infrequent words |
| Unknown words | Map to [UNK] | Handles words missing from the selected fastText vocabulary |
| Padding | Dynamic, per batch | Avoids unnecessary padding |
| Truncation | None currently | Preserves the complete short narratives |

### Text Cleaning

The training dataset was checked for HTML, URLs, emojis,
invisible characters, and Unicode irregularities.

No major formatting issues were found. Apostrophe variants
are normalized, and clearly identifiable corrupted
apostrophes are repaired.

Other corrupted characters are kept unchanged.

### Vocabulary and Embeddings

The vocabulary is constructed using training data only.

Only words available in the pretrained
`fasttext-wiki-news-subwords-300` vocabulary are included.

Final training vocabulary coverage: 95.03%

Final training token coverage: 99.62%

The implementation uses pretrained 300-dimensional vectors
through a fixed vocabulary lookup. It does not perform
full fastText subword inference for unseen words.

Unknown tokens use a shared [UNK] embedding, initialized
as the mean of the available pretrained word vectors.

The embeddings are frozen during training.

### Input Representation

The four text fields are combined as:

O1 [SEP] O2 [SEP] H1 [SEP] H2

Three separator tokens distinguish the fields.

Labels are converted from {1, 2} to {0, 1}.

Sequences are dynamically padded to the longest sequence
in each batch. Their original lengths are also returned
by the DataLoader.

### Limitations

Some ambiguous contractions, such as "he'd" and "it's",
remain unchanged and may map to [UNK].

Splitting possessives into a base word and the token
"possessive" is a heuristic representation. It may not
always reflect the correct grammatical interpretation.

These choices improve embedding coverage but do not
guarantee improved model accuracy.

Preprocessing decisions are based on training data.
Validation data is reserved for model selection, and
test data is not used for preprocessing or tuning.


In [25]:
import importlib
from project1.src import preprocessing

# Reload the updated preprocessing.py
importlib.reload(preprocessing)

# Compare tokenizer outputs across all training texts
mismatches = []

for idx, example in enumerate(dataset["train"]):
    for field in fields:
        notebook_tokens = tokenize_v2(example[field])
        module_tokens = preprocessing.tokenize(example[field])

        if notebook_tokens != module_tokens:
            mismatches.append((idx, field))

print("=== TOKENIZER CONSISTENCY CHECK ===")
print("Mismatches:", len(mismatches))
print("All tokenizers match:", len(mismatches) == 0)

# Verify vocabulary and example encoding
new_vocab = preprocessing.build_vocab(
    dataset["train"], fasttext
)

new_encoded = preprocessing.encode_example(
    dataset["train"][0], new_vocab
)

print("\n=== PIPELINE CONSISTENCY ===")
print("Vocabulary matches:", new_vocab == vocab)
print("Encoding matches:", new_encoded == encoded)

print("\n=== FASTTEXT COVERAGE ===")
print(f"Vocabulary coverage: {vocab_coverage_v2:.2f}%")
print(f"Token coverage: {token_coverage_v2:.2f}%")


=== TOKENIZER CONSISTENCY CHECK ===
Mismatches: 0
All tokenizers match: True

=== PIPELINE CONSISTENCY ===
Vocabulary matches: True
Encoding matches: True

=== FASTTEXT COVERAGE ===
Vocabulary coverage: 95.03%
Token coverage: 99.62%


In [26]:

from collections import Counter
from project1.src.preprocessing import (
    FIELDS, PAD, UNK, SEP, tokenize, build_vocab
)

print("=== DATA LEAKAGE AUDIT ===")

# Build token frequencies from training data only
train_counts = Counter()

for example in dataset["train"]:
    for field in FIELDS:
        train_counts.update(tokenize(example[field]))

train_tokens = set(train_counts)

# Exclude explicitly defined special tokens
special_tokens = {PAD, UNK, SEP}
model_tokens = set(new_vocab) - special_tokens

# 1. Every vocabulary word must occur in training
train_only = model_tokens.issubset(train_tokens)

# 2. Every model word must have a pretrained vector
fasttext_only = all(
    word in fasttext.key_to_index
    for word in model_tokens
)

# 3. Rebuilding from training must reproduce the vocabulary
rebuilt_vocab = build_vocab(
    dataset["train"], fasttext
)

reproducible = rebuilt_vocab == new_vocab

# 4. Check all training words with available embeddings
expected_words = {
    word for word in train_tokens
    if word in fasttext.key_to_index
}

complete = model_tokens == expected_words

print("Vocabulary built from training:", train_only)
print("All words covered by fastText:", fasttext_only)
print("Vocabulary reproducible:", reproducible)
print("All eligible training words included:", complete)

print("\nVocabulary size:", len(new_vocab))
print("Special tokens:", len(special_tokens))
print("Training-derived model words:", len(model_tokens))

assert train_only
assert fasttext_only
assert reproducible
assert complete

print("\nPASS: Vocabulary construction uses training data only.")


=== DATA LEAKAGE AUDIT ===
Vocabulary built from training: True
All words covered by fastText: True
Vocabulary reproducible: True
All eligible training words included: True

Vocabulary size: 15038
Special tokens: 3
Training-derived model words: 15035

PASS: Vocabulary construction uses training data only.


### Data Leakage Verification

The vocabulary construction pipeline was audited to
ensure that it uses training data only.

All verification checks passed:

- Every vocabulary word originates from the training split.
- Every included word is covered by pretrained fastText.
- Rebuilding the vocabulary produces identical token IDs.
- All eligible training words are included.

The final vocabulary contains 15,038 entries:
- 15,035 training-derived words
- 3 special tokens: [PAD], [UNK], [SEP]

The vocabulary is not expanded using validation or test data.

Validation data is reserved for model selection and
hyperparameter tuning. Test data is reserved for final
evaluation.

**Scope:** This audit verifies vocabulary construction.
Training-loop and model-selection leakage checks will
be performed separately.


In [27]:
import torch
from project1.src.embeddings import build_embedding_layer

# Build embedding layer using reusable module
new_embedding_layer = build_embedding_layer(
    new_vocab,
    fasttext
)

print("Embedding shape:", new_embedding_layer.weight.shape)

print("Embeddings frozen:",
      not new_embedding_layer.weight.requires_grad)

print("PAD vector zero:",
      torch.all(new_embedding_layer.weight[0] == 0).item())

# Compare against original embedding layer
print("Matches original embeddings:",
      torch.equal(
          new_embedding_layer.weight,
          embedding_layer.weight
      ))


Embedding shape: torch.Size([15038, 300])
Embeddings frozen: True
PAD vector zero: True
Matches original embeddings: True


In [28]:

import torch

print("=== FASTTEXT EMBEDDING VERIFICATION ===")

pad_id = new_vocab["[PAD]"]
unk_id = new_vocab["[UNK]"]
sep_id = new_vocab["[SEP]"]

weights = new_embedding_layer.weight.detach()

# 1. Verify embedding shape and frozen weights
print("Embedding shape:", tuple(weights.shape))
print("Embedding dimension:", weights.shape[1])
print("Embeddings frozen:",
      not new_embedding_layer.weight.requires_grad)

# 2. Verify special tokens
print("\n=== SPECIAL TOKENS ===")

print("PAD vector zero:",
      torch.all(weights[pad_id] == 0).item())

print("UNK vector nonzero:",
      torch.any(weights[unk_id] != 0).item())

print("SEP vector nonzero:",
      torch.any(weights[sep_id] != 0).item())

# 3. Check UNK initialization
expected_unk = weights[3:].mean(dim=0)

print("UNK equals mean embedding:",
      torch.allclose(
          weights[unk_id],
          expected_unk,
          atol=1e-6
      ))

# 4. Check actual vocabulary fallback
print("\n=== VOCABULARY LOOKUP ===")

test_words = ["panda", "he'd", "it's", "gina"]

for word in test_words:
    token_id = new_vocab.get(word, unk_id)

    print(
        f"{word:10} "
        f"ID: {token_id:6} "
        f"UNK: {token_id == unk_id}"
    )

# 5. Check coverage using actual training vocabulary
total_tokens = sum(word_counts_v2.values())

unknown_tokens = sum(
    count
    for word, count in word_counts_v2.items()
    if word not in new_vocab
)

coverage = 100 * (1 - unknown_tokens / total_tokens)

print("\n=== FINAL COVERAGE ===")
print("Total tokens:", total_tokens)
print("Unknown tokens:", unknown_tokens)
print(f"Token coverage: {coverage:.2f}%")


=== FASTTEXT EMBEDDING VERIFICATION ===
Embedding shape: (15038, 300)
Embedding dimension: 300
Embeddings frozen: True

=== SPECIAL TOKENS ===
PAD vector zero: True
UNK vector nonzero: True
SEP vector nonzero: True
UNK equals mean embedding: True

=== VOCABULARY LOOKUP ===
panda      ID:   3385 UNK: False
he'd       ID:      1 UNK: True
it's       ID:      1 UNK: True
gina       ID:     68 UNK: False

=== FINAL COVERAGE ===
Total tokens: 656675
Unknown tokens: 2516
Token coverage: 99.62%


### fastText Embedding Verification

The implementation uses pretrained
`fasttext-wiki-news-subwords-300` embeddings
loaded through Gensim.

**Verified configuration:**
- Vocabulary size: 15,038 tokens (including special tokens)
- Embedding dimensions: 300
- Embeddings frozen during training
- [PAD]: zero vector
- [UNK]: mean of available pretrained vocabulary embeddings
- [SEP]: nonzero separator embedding

**Training coverage:**
- Total tokens: 656,675
- Known tokens: 654,159
- Unknown tokens: 2,516
- Vocabulary coverage: 95.03%
- Token coverage: 99.62%

**Limitation:**

Although the pretrained model is based on fastText,
this implementation uses fixed word-vector lookup.

It does not compute subword embeddings for unseen words.
Tokens outside the constructed vocabulary map to [UNK].

Therefore, coverage improvements come from preprocessing
and vocabulary matching, not from full fastText
character n-gram inference.


In [29]:
from project1.src.data_loader import create_dataloader

# Create DataLoaders using our new module
new_train_loader = create_dataloader(
    dataset["train"],
    new_vocab,
    batch_size=32,
    shuffle=True
)

new_val_loader = create_dataloader(
    dataset["validation"],
    new_vocab,
    batch_size=32,
    shuffle=False
)

# Inspect a batch
batch = next(iter(new_train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Lengths shape:", batch["lengths"].shape)
print("Labels shape:", batch["labels"].shape)

print("\nBatch size:", len(batch["labels"]))
print("Padding tokens:",
      (batch["input_ids"] == 0).sum().item())

print("\nTraining batches:", len(new_train_loader))
print("Validation batches:", len(new_val_loader))


Input shape: torch.Size([32, 70])
Lengths shape: torch.Size([32])
Labels shape: torch.Size([32])

Batch size: 32
Padding tokens: 895

Training batches: 526
Validation batches: 32


In [30]:
from collections import Counter
import torch

pad_id = new_vocab["[PAD]"]

def verify_dataloader(name, loader, original_data):
    print(f"\n=== {name.upper()} DATALOADER ===")

    original_lengths = []
    padded_tokens = 0
    total_positions = 0
    batch_widths = set()
    observed = Counter()

    # Expected sequences before batching or padding
    expected = Counter()

    for example in original_data:
        encoded = preprocessing.encode_example(
            example, new_vocab
        )
        key = (
            tuple(encoded["input_ids"]),
            encoded["label"]
        )
        expected[key] += 1

    for batch in loader:
        ids = batch["input_ids"]
        lengths = batch["lengths"]
        labels = batch["labels"]

        batch_size, max_len = ids.shape
        batch_widths.add(max_len)

        # 1. Check tensor shapes and lengths
        assert lengths.shape == (batch_size,)
        assert labels.shape == (batch_size,)
        assert torch.all(lengths > 0)
        assert torch.all(lengths <= max_len)

        # 2. Verify dynamic padding
        assert max_len == lengths.max().item()

        # 3. Create expected padding mask
        positions = torch.arange(
            max_len, device=ids.device
        ).unsqueeze(0)

        mask = positions < lengths.to(
            ids.device
        ).unsqueeze(1)

        # All real tokens must be non-PAD
        assert torch.all(ids[mask] != pad_id)

        # All positions after sequence end must be PAD
        assert torch.all(ids[~mask] == pad_id)

        # Check an existing mask, if provided
        if "attention_mask" in batch:
            assert torch.equal(
                batch["attention_mask"].bool(),
                mask
            )

        # 4. Reconstruct sequences to detect truncation
        for seq, length, label in zip(
            ids, lengths, labels
        ):
            length = int(length)

            key = (
                tuple(seq[:length].tolist()),
                int(label)
            )
            observed[key] += 1
            original_lengths.append(length)

        padded_tokens += (~mask).sum().item()
        total_positions += ids.numel()

    # Every example must be preserved exactly
    assert observed == expected

    lengths_tensor = torch.tensor(
        original_lengths, dtype=torch.float
    )

    print("Examples:", len(original_lengths))
    print("Minimum length:", int(lengths_tensor.min()))
    print("Mean length:", round(lengths_tensor.mean().item(), 2))
    print("95th percentile:", int(torch.quantile(
        lengths_tensor, 0.95
    )))
    print("Maximum length:", int(lengths_tensor.max()))

    print("\nPadding percentage:",
          round(100 * padded_tokens / total_positions, 2))

    print("Different batch widths:", len(batch_widths))
    print("Dynamic padding:", len(batch_widths) > 1)
    print("Padding masks: PASS")
    print("No truncation / missing examples: PASS")


verify_dataloader(
    "Train", new_train_loader, dataset["train"]
)

verify_dataloader(
    "Validation", new_val_loader, dataset["validation"]
)



=== TRAIN DATALOADER ===
Examples: 16801
Minimum length: 20
Mean length: 42.09
95th percentile: 55
Maximum length: 92

Padding percentage: 31.83
Different batch widths: 42
Dynamic padding: True
Padding masks: PASS
No truncation / missing examples: PASS

=== VALIDATION DATALOADER ===
Examples: 1000
Minimum length: 23
Mean length: 42.56
95th percentile: 55
Maximum length: 93

Padding percentage: 33.09
Different batch widths: 21
Dynamic padding: True
Padding masks: PASS
No truncation / missing examples: PASS


### Padding and Truncation Verification

Both training and validation DataLoaders were verified.

| Metric | Train | Validation |
|---|---:|---:|
| Examples | 16,801 | 1,000 |
| Mean sequence length | 42.09 | 42.56 |
| 95th percentile | 55 | 55 |
| Maximum length | 92 | 93 |
| Padding percentage | 31.89% | 33.09% |

**Decisions:**
- Dynamic padding to the longest sequence in each batch.
- [PAD] token ID is 0.
- Original sequence lengths are preserved.
- Padding masks can be reconstructed from sequence lengths.
- No truncation is applied.

All examples and token sequences were preserved exactly.

Model-level handling of padded tokens will be verified separately.


In [31]:
from pathlib import Path
import torch

# Create artifacts directory
artifacts_dir = Path.cwd().parent / "artifacts"
artifacts_dir.mkdir(exist_ok=True)

# Save vocabulary and frozen embedding weights
artifact_path = artifacts_dir / "fasttext_art.pt"

torch.save({
    "vocab": new_vocab,
    "embedding_weights": new_embedding_layer.weight.detach().cpu()
}, artifact_path)

print("Saved to:", artifact_path)
print("File size:", round(artifact_path.stat().st_size / 1024**2, 2), "MB")


Saved to: C:\Users\kimbo\Documents\HSLU\Sem 5\NLP\nlp-art-hs26-kim\project1\artifacts\fasttext_art.pt
File size: 17.49 MB
